In [ ]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath("../../../"))
PROJECT_ROOT = "../../../"

import lib.experimentation.experiments as exp
import lib.feature_selection.statistical_methods as stats_fs
import lib.feature_selection.proposed_methods as proposed
import lib.learning_algorithms.classical as classical
import lib.learning_algorithms.domain_aware as domain_aware
import lib.learning_algorithms.ensemble as ensemble

# Load

In [12]:
data_path = os.path.join(
    PROJECT_ROOT,
    "Datasets/MotorImagery/processed/bci_features.csv"
)
df = exp.load_dataset(path=data_path)
df.head()

,subject,session,label,b4_8_mean_0,b4_8_mean_1,b4_8_mean_2,b4_8_mean_3,b4_8_mean_4,b4_8_mean_5,b4_8_mean_6,...,b32_40_bp_32_40_12,b32_40_bp_32_40_13,b32_40_bp_32_40_14,b32_40_bp_32_40_15,b32_40_bp_32_40_16,b32_40_bp_32_40_17,b32_40_bp_32_40_18,b32_40_bp_32_40_19,b32_40_bp_32_40_20,b32_40_bp_32_40_21
0,A01,session1,3,-6.623572e-09,1.198241e-09,-1.310148e-08,-4.286991e-08,-1.016212e-08,4.970633e-09,2.782433e-08,...,8.495567e-07,4.138743e-07,4.190293e-07,4.348321e-07,4.342427e-07,4.786880e-07,3.465733e-07,3.122935e-07,2.734157e-07,4.752810e-07
1,A01,session1,2,1.352518e-08,4.174438e-08,3.356536e-08,1.563808e-08,-6.676323e-09,-1.616462e-08,3.850782e-08,...,1.023019e-06,4.184724e-07,4.728171e-07,5.895769e-07,6.502926e-07,7.089532e-07,4.144287e-07,4.434910e-07,4.301555e-07,5.523525e-07
2,A01,session1,1,-7.791658e-09,2.169236e-08,1.415169e-08,-8.511926e-10,-1.853582e-08,-2.759449e-08,3.046065e-08,...,7.654174e-07,3.021358e-07,3.294150e-07,3.415892e-07,3.418414e-07,4.132072e-07,3.426869e-07,3.372647e-07,3.518021e-07,6.226845e-07
3,A01,session1,0,1.020144e-07,6.938228e-08,8.288897e-08,9.013959e-08,8.987064e-08,7.207847e-08,4.084847e-08,...,7.179327e-07,2.730885e-07,2.241033e-07,2.066232e-07,2.436456e-07,3.333457e-07,2.322823e-07,1.875491e-07,1.910627e-07,2.843472e-07
4,A01,session1,0,-7.194940e-08,-4.020812e-08,-3.824895e-08,-4.503025e-08,-4.848612e-08,-6.042462e-08,-4.592715e-09,...,7.872938e-07,2.731926e-07,3.004725e-07,3.491033e-07,4.206833e-07,5.159278e-07,2.245471e-07,2.390105e-07,2.477437e-07,4.386975e-07


# Parameters

In [13]:
n_features = 256

In [15]:
fs_dict = {
    # ======================================================
    # DIFS
    # ======================================================
    "C-DIFS": {
        "function": proposed.fs_corr,
        "params": {
            "mode": "per_subject",
            "method": "correlation",
            "metric_fn": proposed.metric_dispersion,
            "N_remove": -1,
            "n_features": n_features
        }
    },
    "K-DIFS-linear": {
        "function": proposed.fs_corr,
        "params": {
            "mode": "per_subject",
            "method": "linear",
            "metric_fn": proposed.metric_dispersion,
            "N_remove": -1,
            "n_features": n_features
        }
    },
    "K-DIFS-rbf": {
        "function": proposed.fs_corr,
        "params": {
            "mode": "per_subject",
            "method": "rbf",
            "metric_fn": proposed.metric_dispersion,
            "N_remove": -1,
            "n_features": n_features
        }
    },
    "K-DIFS-cosine": {
        "function": proposed.fs_corr,
        "params": {
            "mode": "per_subject",
            "method": "cosine",
            "metric_fn": proposed.metric_dispersion,
            "N_remove": -1,
            "n_features": n_features
        }
    },

    # ======================================================
    # SSFS
    # ======================================================
    "C-SSFS": {
        "function": proposed.fs_corr,
        "params": {
            "mode": "per_state",
            "method": "correlation",
            "metric_fn": proposed.metric_state_separation,
            "N_remove": -1,
            "n_features": n_features
        }
    },
    "K-SSFS-linear": {
        "function": proposed.fs_corr,
        "params": {
            "mode": "per_state",
            "method": "linear",
            "metric_fn": proposed.metric_state_separation,
            "N_remove": -1,
            "n_features": n_features
        }
    },
    "K-SSFS-rbf": {
        "function": proposed.fs_corr,
        "params": {
            "mode": "per_state",
            "method": "rbf",
            "metric_fn": proposed.metric_state_separation,
            "N_remove": -1,
            "n_features": n_features
        }
    },
    "K-SSFS-cosine": {
        "function": proposed.fs_corr,
        "params": {
            "mode": "per_state",
            "method": "cosine",
            "metric_fn": proposed.metric_state_separation,
            "N_remove": -1,
            "n_features": n_features
        }
    },

    # ======================================================
    # Mean Separation
    # ======================================================
    "C-Mean": {
        "function": proposed.fs_corr,
        "params": {
            "mode": "per_subject_state",
            "method": "correlation",
            "metric_fn": proposed.metric_mean_separation,
            "N_remove": -1,
            "n_features": n_features
        }
    },
    "K-Mean-linear": {
        "function": proposed.fs_corr,
        "params": {
            "mode": "per_subject_state",
            "method": "linear",
            "metric_fn": proposed.metric_mean_separation,
            "N_remove": -1,
            "n_features": n_features
        }
    },
    "K-Mean-rbf": {
        "function": proposed.fs_corr,
        "params": {
            "mode": "per_subject_state",
            "method": "rbf",
            "metric_fn": proposed.metric_mean_separation,
            "N_remove": -1,
            "n_features": n_features
        }
    },
    "K-Mean-cosine": {
        "function": proposed.fs_corr,
        "params": {
            "mode": "per_subject_state",
            "method": "cosine",
            "metric_fn": proposed.metric_mean_separation,
            "N_remove": -1,
            "n_features": n_features
        }
    },

    # ======================================================
    # Wasserstein
    # ======================================================
    "C-Wasserstein": {
        "function": proposed.fs_corr,
        "params": {
            "mode": "per_subject_state",
            "method": "correlation",
            "metric_fn": proposed.metric_wasserstein,
            "N_remove": -1,
            "n_features": n_features
        }
    },
    "K-Wasserstein-linear": {
        "function": proposed.fs_corr,
        "params": {
            "mode": "per_subject_state",
            "method": "linear",
            "metric_fn": proposed.metric_wasserstein,
            "N_remove": -1,
            "n_features": n_features
        }
    },
    "K-Wasserstein-rbf": {
        "function": proposed.fs_corr,
        "params": {
            "mode": "per_subject_state",
            "method": "rbf",
            "metric_fn": proposed.metric_wasserstein,
            "N_remove": -1,
            "n_features": n_features
        }
    },
    "K-Wasserstein-cosine": {
        "function": proposed.fs_corr,
        "params": {
            "mode": "per_subject_state",
            "method": "cosine",
            "metric_fn": proposed.metric_wasserstein,
            "N_remove": -1,
            "n_features": n_features
        }
    },

    # ======================================================
    # Bhattacharyya
    # ======================================================
    "C-Bhattacharyya": {
        "function": proposed.fs_corr,
        "params": {
            "mode": "per_subject_state",
            "method": "correlation",
            "metric_fn": proposed.metric_bhattacharyya,
            "N_remove": -1,
            "n_features": n_features
        }
    },
    "K-Bhattacharyya-linear": {
        "function": proposed.fs_corr,
        "params": {
            "mode": "per_subject_state",
            "method": "linear",
            "metric_fn": proposed.metric_bhattacharyya,
            "N_remove": -1,
            "n_features": n_features
        }
    },
    "K-Bhattacharyya-rbf": {
        "function": proposed.fs_corr,
        "params": {
            "mode": "per_subject_state",
            "method": "rbf",
            "metric_fn": proposed.metric_bhattacharyya,
            "N_remove": -1,
            "n_features": n_features
        }
    },
    "K-Bhattacharyya-cosine": {
        "function": proposed.fs_corr,
        "params": {
            "mode": "per_subject_state",
            "method": "cosine",
            "metric_fn": proposed.metric_bhattacharyya,
            "N_remove": -1,
            "n_features": n_features
        }
    },
}

In [17]:
model_dict = {
    "nn_erm": {
        "function": classical.train_nn_erm,
        "params": {
            "hidden_dim": 128,
            "epochs": 50,
            "lr": 1e-3,
            "batch_size": 128
        }
    }
}

# Experiments

In [ ]:
intra_result_df, intra_time_df = exp.run_experiment(df, exp.split_intra_subject_session, fs_dict, model_dict)
exp.save_results(intra_result_df, intra_time_df, "results_fs/", "intra")

Splits: 100%|██████████| 18/18 [00:03<00:00,  5.95it/s, type=intra_session, subject=A09, session=session2]


In [ ]:
inter_session_result_df, inter_session_time_df = exp.run_experiment(df, exp.split_inter_session, fs_dict, model_dict)
exp.save_results(inter_session_result_df, inter_session_time_df, "results_fs/", "inter_session")

Splits: 100%|██████████| 18/18 [00:03<00:00,  5.62it/s, type=inter_session, subject=A09, session=session2]


In [20]:
inter_subject_result_df, inter_subject_time_df = exp.run_experiment(df, exp.split_inter_subject, fs_dict, model_dict)
exp.save_results(inter_subject_result_df, inter_subject_time_df, "results_fs/", "inter_subject")

Splits: 100%|██████████| 9/9 [00:26<00:00,  2.97s/it, type=inter_subject, subject=A09, session=-]
